In [14]:
!pip install biopython
!pip install transformers torch

In [15]:
import requests
from Bio import SeqIO
from io import StringIO
import torch
from transformers import AutoTokenizer, EsmForMaskedLM

# Carichiamo il modello (usiamo l'8M per velocità)
model_name = "facebook/esm2_t6_8M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmForMaskedLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

EsmForMaskedLM LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     |  | 
----------------------------+------------+--+-
esm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
def fasta_to_clean_string(file_path):
    sequences = []
    # parse gestisce file con una o più proteine
    for record in SeqIO.parse(file_path, "fasta"):
        # record.seq è un oggetto speciale, lo trasformiamo in stringa semplice
        clean_seq = str(record.seq)
        sequences.append(clean_seq)

    return sequences

In [17]:
def importProtein(uniprot_id):

  url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
  response = requests.get(url)

  if response.status_code == 200:
      print(response.text) # Hai la sequenza pronta per l'AI
      print(fasta_to_clean_string(StringIO(response.text)))
      print('lunghezza',len(fasta_to_clean_string(StringIO(response.text))[0]))
      return fasta_to_clean_string(StringIO(response.text))
  else:
    print('nessuna proteina trovata')

In [18]:
def trova_punti_flessibili(sequenza):
    inputs = tokenizer(sequenza, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.softmax(logits, dim=-1)
    tokens = inputs["input_ids"][0]

    punteggi = []
    # Saltiamo <cls> (0) e <eos> (ultimo)
    for i in range(1, len(tokens) - 1):
        token_id = tokens[i]
        prob = probs[0, i, token_id].item()
        punteggi.append((i-1, sequenza[i-1], prob)) # (indice, amminoacido, probabilità)

    # Restituiamo la lista ordinata dal più "scarso" al più "sicuro"
    return sorted(punteggi, key=lambda x: x[2])

In [19]:
StartingProtein=input()
sequence=importProtein(StartingProtein)[0]
print(trova_punti_flessibili(sequence))

P11006
>sp|P11006|MAGA_XENLA Magainins OS=Xenopus laevis OX=8355 GN=magainins PE=1 SV=1
MFKGLFICSLIAVICANALPQPEASADEDMDEREVRGIGKFLHSAGKFGKAFVGEIMKSK
RDAEAVGPEAFADEDLDEREVRGIGKFLHSAKKFGKAFVGEIMNSKRDAEAVGPEAFADE
DLDEREVRGIGKFLHSAKKFGKAFVGEIMNSKRDAEAVGPEAFADEDLDEREVRGIGKFL
HSAKKFGKAFVGEIMNSKRDAEAVGPEAFADEDFDEREVRGIGKFLHSAKKFGKAFVGEI
MNSKRDAEAVGPEAFADEDLDEREVRGIGKFLHSAKKFGKAFVGEIMNSKRDAEAVDDRR
WVE

['MFKGLFICSLIAVICANALPQPEASADEDMDEREVRGIGKFLHSAGKFGKAFVGEIMKSKRDAEAVGPEAFADEDLDEREVRGIGKFLHSAKKFGKAFVGEIMNSKRDAEAVGPEAFADEDLDEREVRGIGKFLHSAKKFGKAFVGEIMNSKRDAEAVGPEAFADEDLDEREVRGIGKFLHSAKKFGKAFVGEIMNSKRDAEAVGPEAFADEDFDEREVRGIGKFLHSAKKFGKAFVGEIMNSKRDAEAVGPEAFADEDLDEREVRGIGKFLHSAKKFGKAFVGEIMNSKRDAEAVDDRRWVE']
lunghezza 303


TypeError: 'NoneType' object is not subscriptable